#  FCN HYPOGLYCAEMIA PREDICTION — Downstream imputation benchmark

This notebook predicts hypoglycaemia (any glucose < 70 mg/dL in the next 30 min) from 2 h of glucose and HR with a 1-D fully convolutional network. The FCN is trained on the original data or on data imputed by BRITS, CSDI-adapted or CAKGI (PIDCK).

**Design**
- **Splits:** 3 repeats × 5 patient-level folds, stratified by each patient's hypoglycaemia burden. Validation patients are taken from the training folds and are identical for every method.
- **Training and validation windows:** these come from each method's imputed series.
- **Labels:** always from originally observed future glucose. Imputed values are never targets.
- **Test windows:** the same fully observed original windows for every method, so imputation only changes the training data.
- **Model selection:** early stopping on validation AUROC; the decision threshold maximises validation F1.
- **Statistics:** Nadeau–Bengio corrected repeated-CV t-tests, with Holm correction across the 9 planned comparisons.

**Inputs**
- `SDBII.csv` and `SDBII_gap_classified.csv`. The latter is written by `CAKGI_run_imputation.ipynb`.
- `SDBII_imputed_brits.csv`, `SDBII_imputed_csdi.csv` and `SDBII_imputed_pidck.csv`, written by the three imputation notebooks.

All files must be on the same patient/timestamp grid; the notebook checks this.

# Imports and constants

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc as sk_auc,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

BASE = Path(r"path/to/your/data_directory")     # <-- set this

ORIGINAL_FILE     = BASE / "SDBII_gap_classified.csv"
ORIGINAL_RAW_FILE = BASE / "SDBII.csv"

METHOD_FILES = {
    'original': None,
    'brits': BASE / "SDBII_imputed_brits.csv",
    'csdi': BASE / "SDBII_imputed_csdi.csv",
    'pidck': BASE / "SDBII_imputed_pidck.csv",
}

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HYPO_THR     = 70.0
WINDOW_STEPS = 24       # 120 minutes
PRED_STEPS   = 6        # following 30 minutes
N_CHANNELS   = 4

BATCH_SIZE   = 512
EPOCHS       = 40
LR           = 5e-4
WEIGHT_DECAY = 1e-3
EARLY_STOP_PATIENCE = 7       

N_SPLITS  = 5
N_REPEATS = 3
SEED      = 42

VAL_FRACTION    = 0.15
MIN_VAL_PATIENTS = 5
MIN_VAL_HYPO_WINDOWS = 30
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.025)

OUTPUT_PREFIX = BASE / "fcn_imputation_benchmark_final"

print(f"Device: {DEVICE}")
print(f"Outer evaluation: {N_REPEATS} repeats x {N_SPLITS} patient folds = "
      f"{N_REPEATS * N_SPLITS} runs")
print(f"Input: {WINDOW_STEPS * 5} min | Prediction horizon: {PRED_STEPS * 5} min")
print(f"Threshold grid: {THRESHOLD_GRID[0]:.3f} to {THRESHOLD_GRID[-1]:.3f}")
print("\nFiles:")
for name, path in METHOD_FILES.items():
    if path is None:
        print(f"  {name:<10}: canonical original observed data")
    else:
        print(f"  {name:<10}: {'OK' if path.exists() else 'NOT FOUND'} — {path}")

# Canonical observed source and grid/value validation

In [ ]:
REQUIRED_ORIGINAL_COLS = {'PtID', 'ts', 'GlucoseCGM', 'HR'}


def _read_and_sort(path, usecols=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, usecols=usecols, low_memory=False)
    if 'ts' not in df.columns or 'PtID' not in df.columns:
        raise ValueError(f"{path.name} must contain PtID and ts")
    df['ts'] = pd.to_datetime(df['ts'], format='mixed', dayfirst=True, errors='raise')
    df = df.sort_values(['PtID', 'ts']).reset_index(drop=True)
    if df.duplicated(['PtID', 'ts']).any():
        examples = df.loc[df.duplicated(['PtID', 'ts'], keep=False), ['PtID', 'ts']].head()
        raise ValueError(f"Duplicate (PtID, ts) rows in {path.name}:\n{examples}")
    return df


def _replace_sensor_zeros_with_nan(df):
    df = df.copy()
    for col in ('GlucoseCGM', 'HR'):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').replace(0, np.nan)
    return df


def assert_same_grid(reference, candidate, reference_name, candidate_name):
    if len(reference) != len(candidate):
        raise ValueError(
            f"Grid mismatch: {reference_name} has {len(reference):,} rows; "
            f"{candidate_name} has {len(candidate):,} rows"
        )

    ref_pid = reference['PtID'].astype(str).to_numpy()
    can_pid = candidate['PtID'].astype(str).to_numpy()
    if not np.array_equal(ref_pid, can_pid):
        bad = int(np.flatnonzero(ref_pid != can_pid)[0])
        raise ValueError(
            f"Patient-order mismatch between {reference_name} and {candidate_name} "
            f"at sorted row {bad}"
        )

    ref_ts = reference['ts'].to_numpy(dtype='datetime64[ns]')
    can_ts = candidate['ts'].to_numpy(dtype='datetime64[ns]')
    if not np.array_equal(ref_ts, can_ts):
        bad = int(np.flatnonzero(ref_ts != can_ts)[0])
        raise ValueError(
            f"Timestamp-grid mismatch between {reference_name} and {candidate_name} "
            f"at sorted row {bad}: {ref_ts[bad]} vs {can_ts[bad]}"
        )

    ref_patients = set(reference['PtID'].astype(str).unique())
    can_patients = set(candidate['PtID'].astype(str).unique())
    if ref_patients != can_patients:
        raise ValueError(
            f"Patient membership mismatch between {reference_name} and {candidate_name}. "
            f"Missing={sorted(ref_patients-can_patients)[:10]}, "
            f"extra={sorted(can_patients-ref_patients)[:10]}"
        )


def assert_sensor_values_equal(reference, candidate, reference_name, candidate_name,
                               columns=('GlucoseCGM', 'HR'), atol=1e-6):
    for col in columns:
        if col not in reference.columns or col not in candidate.columns:
            raise ValueError(f"Cannot compare {col}: missing in one source")
        a = pd.to_numeric(reference[col], errors='coerce').to_numpy(dtype=float)
        b = pd.to_numeric(candidate[col], errors='coerce').to_numpy(dtype=float)
        if not np.allclose(a, b, rtol=0.0, atol=atol, equal_nan=True):
            finite = np.isfinite(a) & np.isfinite(b)
            diff = np.abs(a - b)
            bad_mask = ((np.isnan(a) != np.isnan(b)) |
                        (finite & (diff > atol)))
            bad = int(np.flatnonzero(bad_mask)[0])
            raise ValueError(
                f"Observed {col} mismatch between {reference_name} and {candidate_name} "
                f"at sorted row {bad}: {a[bad]} vs {b[bad]}"
            )


print("Loading canonical observed/classified source...")
CANONICAL_DF = _read_and_sort(ORIGINAL_FILE)
missing = REQUIRED_ORIGINAL_COLS - set(CANONICAL_DF.columns)
if missing:
    raise ValueError(f"{ORIGINAL_FILE.name} is missing columns: {sorted(missing)}")
CANONICAL_DF = _replace_sensor_zeros_with_nan(CANONICAL_DF)

print("Verifying raw and classified original files use the same patient/timestamp grid and sensors...")
raw_check = _read_and_sort(
    ORIGINAL_RAW_FILE,
    usecols=['PtID', 'ts', 'GlucoseCGM', 'HR']
)
raw_check = _replace_sensor_zeros_with_nan(raw_check)
assert_same_grid(CANONICAL_DF, raw_check,
                 ORIGINAL_FILE.name, ORIGINAL_RAW_FILE.name)
assert_sensor_values_equal(CANONICAL_DF, raw_check,
                           ORIGINAL_FILE.name, ORIGINAL_RAW_FILE.name)
del raw_check

ALL_PIDS = np.array(sorted(CANONICAL_DF['PtID'].unique(), key=str), dtype=object)
CANONICAL_PATIENT_SET = set(CANONICAL_DF['PtID'].astype(str).unique())

# Contiguous row ranges for the canonical sorted grid.
PID_RANGES = {}
for pid, idx in CANONICAL_DF.groupby('PtID', sort=False).indices.items():
    idx = np.asarray(idx)
    if len(idx) and not np.array_equal(idx, np.arange(idx[0], idx[-1] + 1)):
        raise RuntimeError(f"Rows for patient {pid} are not contiguous after sorting")
    PID_RANGES[pid] = (int(idx[0]), int(idx[-1]) + 1)


def get_fold_norm_stats(train_pids):
    fold_df = CANONICAL_DF[CANONICAL_DF['PtID'].isin(train_pids)]
    if fold_df.empty:
        raise RuntimeError("No canonical rows for inner-training patients")

    g_mean = float(fold_df['GlucoseCGM'].mean())
    g_std  = float(fold_df['GlucoseCGM'].std())
    h_mean = float(fold_df['HR'].mean())
    h_std  = float(fold_df['HR'].std())

    if not np.isfinite(g_mean) or not np.isfinite(h_mean):
        raise RuntimeError("Non-finite fold mean")
    if not np.isfinite(g_std) or g_std <= 0:
        g_std = 1.0
    if not np.isfinite(h_std) or h_std <= 0:
        h_std = 1.0

    return {'g_mean': g_mean, 'g_std': g_std,
            'h_mean': h_mean, 'h_std': h_std}

print(f"Canonical validation passed: {len(CANONICAL_DF):,} rows, "
      f"{len(ALL_PIDS)} patients")

# Method loading, strict imputed-column use, and observed-label windows

In [ ]:
CANONICAL_G = CANONICAL_DF['GlucoseCGM'].to_numpy(dtype=float)
CANONICAL_H = CANONICAL_DF['HR'].to_numpy(dtype=float)
CANONICAL_TS = CANONICAL_DF['ts'].to_numpy(dtype='datetime64[ns]')


def _validate_observed_values_preserved(input_g, input_h, method_name,
                                        g_atol=1e-3, h_atol=1e-3):
    """Imputation must not alter originally observed sensor values."""
    g_mask = np.isfinite(CANONICAL_G)
    h_mask = np.isfinite(CANONICAL_H)

    if not np.allclose(input_g[g_mask], CANONICAL_G[g_mask],
                       rtol=1e-6, atol=g_atol, equal_nan=False):
        delta = np.abs(input_g[g_mask] - CANONICAL_G[g_mask])
        raise ValueError(
            f"{method_name}: imputed glucose column changes originally observed values; "
            f"max absolute difference={np.nanmax(delta):.6g}"
        )
    if not np.allclose(input_h[h_mask], CANONICAL_H[h_mask],
                       rtol=1e-6, atol=h_atol, equal_nan=False):
        delta = np.abs(input_h[h_mask] - CANONICAL_H[h_mask])
        raise ValueError(
            f"{method_name}: imputed HR column changes originally observed values; "
            f"max absolute difference={np.nanmax(delta):.6g}"
        )


def load_method_inputs(method_name, file_path):
    """
    Return the exact input glucose/HR arrays used by the classifier.

    ORIGINAL uses canonical observed sensors.
    Every imputed method use *_imputed_final columns.
    """
    if method_name == 'original':
        return CANONICAL_G.copy(), CANONICAL_H.copy()

    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(path)

    header = pd.read_csv(path, nrows=0).columns.tolist()
    required = {'PtID', 'ts', 'GlucoseCGM_imputed_final', 'HR_imputed_final'}
    missing = required - set(header)
    if missing:
        raise ValueError(
            f"{method_name}: {path.name} is missing required imputed columns: "
            f"{sorted(missing)}"
        )

    d = _read_and_sort(
        path,
        usecols=['PtID', 'ts', 'GlucoseCGM_imputed_final', 'HR_imputed_final']
    )
    assert_same_grid(CANONICAL_DF, d, ORIGINAL_FILE.name, path.name)

    input_g = pd.to_numeric(d['GlucoseCGM_imputed_final'], errors='coerce').to_numpy(float)
    input_h = pd.to_numeric(d['HR_imputed_final'], errors='coerce').to_numpy(float)

    _validate_observed_values_preserved(input_g, input_h, method_name)

    g_recovered = int((~np.isfinite(CANONICAL_G) & np.isfinite(input_g)).sum())
    h_recovered = int((~np.isfinite(CANONICAL_H) & np.isfinite(input_h)).sum())
    print(f"  {method_name}: forced inputs = GlucoseCGM_imputed_final / HR_imputed_final")
    print(f"    recovered glucose values={g_recovered:,}; HR values={h_recovered:,}; "
          f"remaining NaN G={np.isnan(input_g).sum():,}, H={np.isnan(input_h).sum():,}")
    del d
    return input_g, input_h


def build_observed_label_raw_windows(input_g, input_h, method_name):
    """
    Build method-specific INPUT windows, but derive every target from canonical
    originally observed future glucose.

    Returned channels are raw-scale:
      0 glucose, 1 HR, 2 glucose difference, 3 glucose acceleration.
    They are normalised separately inside each fold using inner-training stats.
    """
    if len(input_g) != len(CANONICAL_DF) or len(input_h) != len(CANONICAL_DF):
        raise ValueError(f"{method_name}: input array length does not match canonical grid")

    X, y, groups, start_times = [], [], [], []
    skipped_input_nan = 0
    skipped_unobserved_target = 0

    for pid, (start, end) in PID_RANGES.items():
        g = input_g[start:end]
        h = input_h[start:end]
        target_g = CANONICAL_G[start:end]
        ts = CANONICAL_TS[start:end]
        n = end - start
        if n < WINDOW_STEPS + PRED_STEPS:
            continue

        # Derivatives use only past/current input values. Any derivative made
        # undefined by a previous missing value is set to zero, as in the prior code.
        g_delta = np.zeros_like(g, dtype=float)
        g_delta[1:] = np.diff(g)
        g_accel = np.zeros_like(g, dtype=float)
        g_accel[1:] = np.diff(g_delta)
        g_delta = np.nan_to_num(g_delta, nan=0.0, posinf=0.0, neginf=0.0)
        g_accel = np.nan_to_num(g_accel, nan=0.0, posinf=0.0, neginf=0.0)

        for i in range(n - WINDOW_STEPS - PRED_STEPS + 1):
            win_g = g[i:i + WINDOW_STEPS]
            win_h = h[i:i + WINDOW_STEPS]
            if np.isnan(win_g).any() or np.isnan(win_h).any():
                skipped_input_nan += 1
                continue

            # Target is real/original. Imputed future values are never labels.
            horizon = target_g[i + WINDOW_STEPS:
                               i + WINDOW_STEPS + PRED_STEPS]
            if np.isnan(horizon).any():
                skipped_unobserved_target += 1
                continue

            win = np.stack([
                win_g,
                win_h,
                g_delta[i:i + WINDOW_STEPS],
                g_accel[i:i + WINDOW_STEPS],
            ], axis=1)
            X.append(win)
            y.append(int(np.any(horizon < HYPO_THR)))
            groups.append(pid)
            start_times.append(ts[i])

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    groups = np.asarray(groups)
    start_times = np.asarray(start_times, dtype='datetime64[ns]')

    if len(X) == 0:
        raise RuntimeError(f"{method_name}: no eligible windows")

    print(f"  [{method_name.upper()} INPUT WINDOWS] {len(X):,} windows | "
          f"hypo={int(y.sum()):,} ({100*y.mean():.2f}%) | "
          f"input-NaN skips={skipped_input_nan:,} | "
          f"unobserved-target skips={skipped_unobserved_target:,}")
    return X, y, groups, start_times


def normalise_raw_windows(X_raw, stats):
    if len(X_raw) == 0:
        return np.empty_like(X_raw, dtype=np.float32)
    X = X_raw.astype(np.float32, copy=True)
    X[:, :, 0] = (X[:, :, 0] - stats['g_mean']) / stats['g_std']
    X[:, :, 1] = (X[:, :, 1] - stats['h_mean']) / stats['h_std']
    X[:, :, 2] = X[:, :, 2] / stats['g_std']
    X[:, :, 3] = X[:, :, 3] / stats['g_std']
    if not np.isfinite(X).all():
        raise RuntimeError("Non-finite values after fold normalisation")
    return X


# Canonical ORIGINAL windows simultaneously provide:
#   1) the ORIGINAL method's eligible training/validation pool; and
#   2) the one fixed, fully observed test-window pool for every method.
print("\nBuilding canonical observed input/observed target window pool once...")
ORIGINAL_RAW_X, ORIGINAL_Y, ORIGINAL_GROUPS, ORIGINAL_START_TIMES = \
    build_observed_label_raw_windows(CANONICAL_G, CANONICAL_H, 'original')

# The test pool is canonical and never rebuilt from an imputed dataframe.
CANONICAL_TEST_RAW_X = ORIGINAL_RAW_X
CANONICAL_TEST_Y = ORIGINAL_Y
CANONICAL_TEST_GROUPS = ORIGINAL_GROUPS
CANONICAL_TEST_START_TIMES = ORIGINAL_START_TIMES

print("Canonical test pool is fixed and will be reused exactly for all methods.")

# Standard 1D Fully Convolutional Network and training

In [ ]:
class HypoDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32).permute(0, 2, 1)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class FCN1D(nn.Module):
    def __init__(self, in_channels=N_CHANNELS):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 128, kernel_size=8, padding='same', bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, padding='same', bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 128, kernel_size=3, padding='same', bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        self.global_average_pool = nn.AdaptiveAvgPool1d(1)
        self.output = nn.Linear(128, 1)

    def forward(self, x):
        h = self.features(x)
        h = self.global_average_pool(h).squeeze(-1)
        return self.output(h).squeeze(1)


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.5):
        super().__init__()
        self.alpha = float(alpha)
        self.gamma = float(gamma)

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        p_t = p * targets + (1.0 - p) * (1.0 - targets)
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        return (alpha_t * (1.0 - p_t).pow(self.gamma) * bce).mean()


def find_best_threshold(probs, labels, thresholds=THRESHOLD_GRID):
    best_threshold = 0.5
    best_f1 = -np.inf
    for threshold in thresholds:
        preds = (probs >= threshold).astype(int)
        score = f1_score(labels, preds, pos_label=1, zero_division=0)
        if score > best_f1 + 1e-12:
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold


def binary_metrics(labels, probs, threshold, prefix):
    labels = np.asarray(labels, dtype=int)
    probs = np.asarray(probs, dtype=float)
    preds = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    result = {
        f'{prefix}_prec_hypo': precision_score(labels, preds, pos_label=1, zero_division=0),
        f'{prefix}_recall_hypo': recall_score(labels, preds, pos_label=1, zero_division=0),
        f'{prefix}_f1_hypo': f1_score(labels, preds, pos_label=1, zero_division=0),
        f'{prefix}_prec_normal': precision_score(labels, preds, pos_label=0, zero_division=0),
        f'{prefix}_recall_normal': recall_score(labels, preds, pos_label=0, zero_division=0),
        f'{prefix}_f1_normal': f1_score(labels, preds, pos_label=0, zero_division=0),
        f'{prefix}_specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        f'{prefix}_n_windows': int(len(labels)),
        f'{prefix}_n_hypo': int(labels.sum()),
        f'{prefix}_n_normal': int((1 - labels).sum()),
    }

    if len(np.unique(labels)) == 2 and np.isfinite(probs).all():
        result[f'{prefix}_auroc'] = roc_auc_score(labels, probs)
        precision, recall, _ = precision_recall_curve(labels, probs)
        result[f'{prefix}_pr_auc'] = sk_auc(recall, precision)
        result[f'{prefix}_ap'] = average_precision_score(labels, probs)
    else:
        result[f'{prefix}_auroc'] = np.nan
        result[f'{prefix}_pr_auc'] = np.nan
        result[f'{prefix}_ap'] = np.nan
    return result


def predict_loader(model, loader):
    model.eval()
    probabilities, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE))
            probabilities.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(yb.numpy())
    return np.asarray(probabilities), np.asarray(labels)


def _set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def train_and_eval(X_train, y_train, X_val, y_val, X_test, y_test, seed):
    """
    Early stopping model criterion: validation AUROC.
    Threshold selection: validation F1 at the selected model checkpoint.
    Test is evaluated once with that frozen checkpoint and threshold.
    """
    _set_seed(seed)

    for split_name, labels in [('train', y_train), ('validation', y_val), ('test', y_test)]:
        if len(labels) == 0 or len(np.unique(labels)) != 2:
            raise RuntimeError(f"{split_name} must be non-empty and contain both classes")

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        HypoDataset(X_train, y_train), batch_size=BATCH_SIZE,
        shuffle=True, generator=generator, num_workers=0
    )
    train_eval_loader = DataLoader(
        HypoDataset(X_train, y_train), batch_size=BATCH_SIZE,
        shuffle=False, num_workers=0
    )
    val_loader = DataLoader(
        HypoDataset(X_val, y_val), batch_size=BATCH_SIZE,
        shuffle=False, num_workers=0
    )
    test_loader = DataLoader(
        HypoDataset(X_test, y_test), batch_size=BATCH_SIZE,
        shuffle=False, num_workers=0
    )

    model = FCN1D().to(DEVICE)
    positive_rate = float(np.mean(y_train))
    criterion = FocalLoss(alpha=max(0.5, 1.0 - positive_rate), gamma=2.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6
    )

    best_state = None
    best_val_auroc = -np.inf
    best_threshold = 0.5
    best_epoch = 0
    best_train_loss = np.nan
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_seen = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += float(loss.item()) * len(yb)
            total_seen += len(yb)

        train_loss = total_loss / max(total_seen, 1)
        val_probs, val_labels = predict_loader(model, val_loader)
        val_auroc = roc_auc_score(val_labels, val_probs)
        val_ap = average_precision_score(val_labels, val_probs)
        scheduler.step(val_auroc)

        history.append({
            'epoch': epoch,
            'train_loss': float(train_loss),
            'val_auroc': float(val_auroc),
            'val_ap': float(val_ap),
            'learning_rate': float(optimizer.param_groups[0]['lr']),
        })

        if val_auroc > best_val_auroc + 1e-6:
            best_val_auroc = float(val_auroc)
            best_threshold = find_best_threshold(val_probs, val_labels)
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            best_epoch = epoch
            best_train_loss = float(train_loss)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                break

    if best_state is None:
        raise RuntimeError("No valid model checkpoint was selected")

    model.load_state_dict(best_state)
    train_probs, train_labels = predict_loader(model, train_eval_loader)
    val_probs, val_labels = predict_loader(model, val_loader)
    test_probs, test_labels = predict_loader(model, test_loader)

    result = {
        'best_epoch': int(best_epoch),
        'epochs_ran': int(history[-1]['epoch']),
        'best_train_loss': float(best_train_loss),
        'best_val_auroc_for_early_stopping': float(best_val_auroc),
        'threshold_used': float(best_threshold),
        'threshold_at_lower_boundary': bool(np.isclose(best_threshold, THRESHOLD_GRID[0])),
        'threshold_at_upper_boundary': bool(np.isclose(best_threshold, THRESHOLD_GRID[-1])),
        'history': history,
    }
    result.update(binary_metrics(train_labels, train_probs, best_threshold, 'train'))
    result.update(binary_metrics(val_labels, val_probs, best_threshold, 'val'))
    result.update(binary_metrics(test_labels, test_probs, best_threshold, 'test'))

    # Compact aliases used in summary cells.
    result.update({
        'auroc': result['test_auroc'],
        'pr_auc': result['test_pr_auc'],
        'ap': result['test_ap'],
        'prec_hypo': result['test_prec_hypo'],
        'recall_hypo': result['test_recall_hypo'],
        'f1_hypo': result['test_f1_hypo'],
        'prec_normal': result['test_prec_normal'],
        'recall_normal': result['test_recall_normal'],
        'f1_normal': result['test_f1_normal'],
        'sensitivity': result['test_recall_hypo'],
        'specificity': result['test_specificity'],
    })
    return result

print("Standard FCN ready: Conv(128/8) -> Conv(256/5) -> Conv(128/3) -> GAP.")

# Patient-level outer splits stratified by observed test burden

In [ ]:
def hypo_tier(count):
    if count >= 200:
        return 2   # high
    if count >= 30:
        return 1   # medium
    return 0       # low


pid_hypo_counts = {}
pid_total_counts = {}
for pid in ALL_PIDS:
    mask = ORIGINAL_GROUPS == pid
    pid_total_counts[pid] = int(mask.sum())
    pid_hypo_counts[pid] = int(ORIGINAL_Y[mask].sum())

strat_labels = np.asarray([
    hypo_tier(pid_hypo_counts.get(pid, 0)) for pid in ALL_PIDS
], dtype=int)

tier_counts = {tier: int((strat_labels == tier).sum()) for tier in np.unique(strat_labels)}
print("Patient burden tiers:", tier_counts)
for tier, count in tier_counts.items():
    if count < N_SPLITS:
        raise ValueError(
            f"Hypoglycaemia tier {tier} contains only {count} patients, fewer than "
            f"N_SPLITS={N_SPLITS}. Revise tier thresholds or reduce folds."
        )

OUTER_SPLITS = []
for repeat_index in range(N_REPEATS):
    repeat_seed = SEED + repeat_index
    splitter = StratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=repeat_seed
    )
    print(f"\nRepeat {repeat_index + 1}, seed={repeat_seed}")
    for fold_index, (train_idx, test_idx) in enumerate(
            splitter.split(ALL_PIDS, strat_labels), start=1):
        train_pids = set(ALL_PIDS[train_idx])
        test_pids = set(ALL_PIDS[test_idx])
        if not train_pids.isdisjoint(test_pids):
            raise RuntimeError("Outer train/test patient overlap")

        outer_run = repeat_index * N_SPLITS + fold_index
        test_mask = np.isin(ORIGINAL_GROUPS, list(test_pids))
        test_y = ORIGINAL_Y[test_mask]
        if len(test_y) == 0 or len(np.unique(test_y)) != 2:
            raise RuntimeError(
                f"Outer run {outer_run}: canonical test set lacks both classes"
            )

        tiers = [hypo_tier(pid_hypo_counts[p]) for p in test_pids]
        OUTER_SPLITS.append({
            'outer_run': outer_run,
            'repeat': repeat_index + 1,
            'fold_in_repeat': fold_index,
            'repeat_seed': repeat_seed,
            'outer_train_pids': train_pids,
            'test_pids': test_pids,
        })
        print(
            f"  run {outer_run:>2}: test={len(test_pids)} patients | "
            f"H={tiers.count(2)} M={tiers.count(1)} L={tiers.count(0)} | "
            f"canonical test windows={len(test_y):,} | hypo={int(test_y.sum()):,}"
        )

if len(OUTER_SPLITS) != N_REPEATS * N_SPLITS:
    raise RuntimeError("Incorrect number of outer splits")

print(f"\nCreated {len(OUTER_SPLITS)} patient-level outer evaluations.")
print("Stratification promotes balanced burden across folds; it does not imply independence.")

# One common patient-level inner validation assignment per outer run

In [ ]:
def split_inner_train_val_patients(outer_train_pids, patient_stats, seed,
                                   val_fraction=VAL_FRACTION,
                                   min_val_patients=MIN_VAL_PATIENTS,
                                   min_hypo_windows=MIN_VAL_HYPO_WINDOWS,
                                   max_attempts=30000):
    """Choose validation patients using ORIGINAL observed-input/observed-label windows."""
    available = np.array(sorted(patient_stats.keys(), key=str), dtype=object)
    if len(available) < 3:
        raise RuntimeError("Fewer than three ORIGINAL-contributing development patients")

    n_val = max(min_val_patients, int(np.ceil(val_fraction * len(outer_train_pids))))
    n_val = min(n_val, len(available) - 1)
    if n_val < 1:
        raise RuntimeError("Cannot allocate validation patients")

    rng = np.random.RandomState(seed)
    best = None
    best_score = -np.inf

    def aggregate(pids):
        total = sum(patient_stats[p]['total'] for p in pids)
        hypo = sum(patient_stats[p]['hypo'] for p in pids)
        normal = total - hypo
        tiers = [hypo_tier(patient_stats[p]['hypo']) for p in pids]
        return total, hypo, normal, tiers

    for _ in range(max_attempts):
        val_pids = set(rng.choice(available, size=n_val, replace=False).tolist())
        original_train_contributors = set(available.tolist()) - val_pids

        v_total, v_hypo, v_normal, v_tiers = aggregate(val_pids)
        t_total, t_hypo, t_normal, _ = aggregate(original_train_contributors)
        if min(v_hypo, v_normal, t_hypo, t_normal) <= 0:
            continue

        score = 0.0
        score += 100000.0 if v_hypo >= min_hypo_windows else 0.0
        score += 1000.0 * len(set(v_tiers))
        score += min(v_hypo, 500)
        score += min(v_total, 20000) / 20000.0

        if score > best_score:
            best_score = score
            best = (val_pids, v_total, v_hypo, v_normal, v_tiers)

    if best is None:
        raise RuntimeError(
            "Could not construct a patient-disjoint, two-class validation set. "
            "Increase validation size or inspect patient-level positive counts."
        )
    return best


FOLD_CONFIGS = {}
for split in OUTER_SPLITS:
    outer_run = split['outer_run']
    outer_train_pids = set(split['outer_train_pids'])
    test_pids = set(split['test_pids'])

    dev_mask = np.isin(ORIGINAL_GROUPS, list(outer_train_pids))
    dev_y = ORIGINAL_Y[dev_mask]
    dev_groups = ORIGINAL_GROUPS[dev_mask]

    patient_stats = {}
    for pid in np.unique(dev_groups):
        mask = dev_groups == pid
        patient_stats[pid] = {
            'total': int(mask.sum()),
            'hypo': int(dev_y[mask].sum()),
        }

    val_pids, v_total, v_hypo, v_normal, v_tiers = \
        split_inner_train_val_patients(
            outer_train_pids, patient_stats,
            seed=SEED + outer_run - 1
        )
    val_pids = set(val_pids)

    # Every other outer-development patient remains assigned to training.
    train_pids = outer_train_pids - val_pids
    if not train_pids.isdisjoint(val_pids):
        raise RuntimeError("Inner train/validation overlap")
    if not train_pids.isdisjoint(test_pids):
        raise RuntimeError("Inner train/test overlap")
    if not val_pids.isdisjoint(test_pids):
        raise RuntimeError("Validation/test overlap")
    if train_pids | val_pids != outer_train_pids:
        raise RuntimeError("Inner split does not cover outer development patients")

    norm_stats = get_fold_norm_stats(train_pids)
    test_indices = np.flatnonzero(np.isin(CANONICAL_TEST_GROUPS, list(test_pids)))
    if len(test_indices) == 0 or len(np.unique(CANONICAL_TEST_Y[test_indices])) != 2:
        raise RuntimeError(f"Outer run {outer_run}: invalid canonical test subset")

    FOLD_CONFIGS[outer_run] = {
        **split,
        'train_pids': train_pids,
        'val_pids': val_pids,
        'norm_stats': norm_stats,
        'canonical_test_indices': test_indices,
    }

    zero_original_train_contributors = len(train_pids - set(patient_stats))
    print(
        f"run {outer_run:>2}: train={len(train_pids)} patients | "
        f"val={len(val_pids)} patients ({v_total:,} original windows; "
        f"hypo={v_hypo:,}, normal={v_normal:,}; "
        f"H={v_tiers.count(2)} M={v_tiers.count(1)} L={v_tiers.count(0)}) | "
        f"train IDs with zero ORIGINAL windows={zero_original_train_contributors}"
    )

if len(FOLD_CONFIGS) != len(OUTER_SPLITS):
    raise RuntimeError("Common inner split construction incomplete")

print("\nCommon patient assignments and canonical test indices are fixed for all methods.")

# Preflight every imputed file before any model training

In [ ]:
print("Preflight: verifying all imputed files before the long experiment...")
for method_name, file_path in METHOD_FILES.items():
    if method_name == 'original':
        continue
    g, h = load_method_inputs(method_name, file_path)
    del g, h
print("Preflight passed: grids, patient membership, columns, and observed-value preservation are valid.")

# Main downstream benchmark

In [ ]:
def _mean_std(fold_results, key):
    values = np.asarray([r[key] for r in fold_results if np.isfinite(r.get(key, np.nan))])
    if len(values) == 0:
        return np.nan, np.nan
    return float(values.mean()), float(values.std(ddof=0))


all_results = {}

for method_index, (method_name, file_path) in enumerate(METHOD_FILES.items()):
    print("\n" + "=" * 100)
    print(f"METHOD: {method_name.upper()}")
    print("=" * 100)

    if method_name == 'original':
        method_raw_X = ORIGINAL_RAW_X
        method_y = ORIGINAL_Y
        method_groups = ORIGINAL_GROUPS
        method_start_times = ORIGINAL_START_TIMES
    else:
        input_g, input_h = load_method_inputs(method_name, file_path)
        method_raw_X, method_y, method_groups, method_start_times = \
            build_observed_label_raw_windows(input_g, input_h, method_name)
        del input_g, input_h

    fold_results = []
    print(f"\n  {'Run':>3} {'Rep':>3} {'Fold':>4} {'Train':>9} {'Val':>9} {'Test':>9} "
          f"{'AUROC':>7} {'PR-AUC':>7} {'P_hypo':>8} {'R_hypo':>8} "
          f"{'F1_hypo':>8} {'Spec':>7} {'Thr':>6} {'Epoch':>6}")
    print("  " + "-" * 116)

    for outer_run in sorted(FOLD_CONFIGS):
        cfg = FOLD_CONFIGS[outer_run]
        train_pids = cfg['train_pids']
        val_pids = cfg['val_pids']
        test_pids = cfg['test_pids']
        stats = cfg['norm_stats']

        train_mask = np.isin(method_groups, list(train_pids))
        val_mask = np.isin(method_groups, list(val_pids))
        test_indices = cfg['canonical_test_indices']

        X_train = normalise_raw_windows(method_raw_X[train_mask], stats)
        y_train = method_y[train_mask]
        X_val = normalise_raw_windows(method_raw_X[val_mask], stats)
        y_val = method_y[val_mask]

        # EXACTLY the same canonical observed test data for every method.
        X_test = normalise_raw_windows(CANONICAL_TEST_RAW_X[test_indices], stats)
        y_test = CANONICAL_TEST_Y[test_indices]
        test_groups = CANONICAL_TEST_GROUPS[test_indices]
        test_start_times = CANONICAL_TEST_START_TIMES[test_indices]

        for split_name, labels in [('train', y_train), ('validation', y_val), ('test', y_test)]:
            if len(labels) == 0 or len(np.unique(labels)) != 2:
                raise RuntimeError(
                    f"{method_name}, outer run {outer_run}: {split_name} is empty "
                    f"or lacks both classes. No fallback is permitted."
                )

        metrics = train_and_eval(
            X_train, y_train, X_val, y_val, X_test, y_test,
            seed=SEED + outer_run  # identical model seed across methods within this outer run
        )
        metrics.update({
            'method': method_name,
            'outer_run': outer_run,
            'repeat': cfg['repeat'],
            'fold_in_repeat': cfg['fold_in_repeat'],
            'repeat_seed': cfg['repeat_seed'],
            'model_seed': SEED + outer_run,
            'n_train_patients': len(train_pids),
            'n_val_patients': len(val_pids),
            'n_test_patients': len(test_pids),
            'train_patient_ids': '|'.join(map(str, sorted(train_pids, key=str))),
            'val_patient_ids': '|'.join(map(str, sorted(val_pids, key=str))),
            'test_patient_ids': '|'.join(map(str, sorted(test_pids, key=str))),
            'test_data_source': 'canonical_original_observed',
            'targets_source': 'canonical_original_observed_future_glucose',
            'n_method_train_windows': int(len(y_train)),
            'n_method_val_windows': int(len(y_val)),
            'n_canonical_test_windows': int(len(y_test)),
        })
        fold_results.append(metrics)

        print(
            f"  {outer_run:>3} {cfg['repeat']:>3} {cfg['fold_in_repeat']:>4} "
            f"{len(y_train):>9,} {len(y_val):>9,} {len(y_test):>9,} "
            f"{metrics['auroc']:>7.4f} {metrics['pr_auc']:>7.4f} "
            f"{metrics['prec_hypo']:>8.4f} {metrics['recall_hypo']:>8.4f} "
            f"{metrics['f1_hypo']:>8.4f} {metrics['specificity']:>7.4f} "
            f"{metrics['threshold_used']:>6.3f} {metrics['best_epoch']:>6}"
        )
        print(
            f"      train AUC={metrics['train_auroc']:.4f}, F1={metrics['train_f1_hypo']:.4f} | "
            f"val AUC={metrics['val_auroc']:.4f}, PR-AUC={metrics['val_pr_auc']:.4f}, "
            f"F1={metrics['val_f1_hypo']:.4f} | epochs ran={metrics['epochs_ran']}"
        )

    completed = {r['outer_run'] for r in fold_results}
    expected = set(FOLD_CONFIGS)
    if completed != expected:
        raise RuntimeError(
            f"{method_name}: incomplete runs. Missing={sorted(expected-completed)}, "
            f"unexpected={sorted(completed-expected)}"
        )

    all_results[method_name] = fold_results

    print("  " + "-" * 116)
    for key, label in [
        ('auroc', 'AUROC'), ('pr_auc', 'PR-AUC'), ('f1_hypo', 'F1 hypo'),
        ('recall_hypo', 'Recall hypo'), ('prec_hypo', 'Precision hypo'),
        ('specificity', 'Specificity')
    ]:
        mean, std = _mean_std(fold_results, key)
        print(f"  {label:<16}: {mean:.4f} +/- {std:.4f}")

    boundary_count = sum(
        r['threshold_at_lower_boundary'] or r['threshold_at_upper_boundary']
        for r in fold_results
    )
    if boundary_count:
        print(f"  WARNING: {boundary_count} thresholds selected a search boundary.")

    if method_name != 'original':
        del method_raw_X, method_y, method_groups, method_start_times

print("\nAll methods completed the same 15 aligned outer evaluations.")

# Final test summary and per-class table

In [ ]:
METRICS_TO_SHOW = [
    ('auroc', 'AUROC'),
    ('pr_auc', 'PR-AUC'),
    ('ap', 'Average precision'),
    ('prec_hypo', 'Precision hypo'),
    ('recall_hypo', 'Recall hypo'),
    ('f1_hypo', 'F1 hypo'),
    ('prec_normal', 'Precision normal'),
    ('recall_normal', 'Recall normal'),
    ('f1_normal', 'F1 normal'),
    ('specificity', 'Specificity'),
]

print("\n" + "=" * 110)
print("FINAL TEST RESULTS — same canonical observed test windows for every method")
print("=" * 110)

header = f"{'Method':<12}" + ''.join(f"{label:>20}" for _, label in METRICS_TO_SHOW)
print(header)
print('-' * len(header))

summary = {}
for method_name, fold_results in all_results.items():
    row = f"{method_name:<12}"
    summary[method_name] = {}
    for key, label in METRICS_TO_SHOW:
        mean, std = _mean_std(fold_results, key)
        summary[method_name][key] = (mean, std)
        row += f"{f'{mean:.3f}+/-{std:.3f}':>20}"
    print(row)

print("\nPer-class precision / recall / F1")
print(f"{'Method':<12} {'Class':<12} {'Precision':>18} {'Recall':>18} {'F1':>18}")
print('-' * 80)
for method_name, fold_results in all_results.items():
    for class_name, prefix in [('Hypo (1)', 'hypo'), ('Normal (0)', 'normal')]:
        p_mean, p_std = _mean_std(fold_results, f'prec_{prefix}')
        r_mean, r_std = _mean_std(fold_results, f'recall_{prefix}')
        f_mean, f_std = _mean_std(fold_results, f'f1_{prefix}')
        p_text = f"{p_mean:.3f}+/-{p_std:.3f}"
        r_text = f"{r_mean:.3f}+/-{r_std:.3f}"
        f_text = f"{f_mean:.3f}+/-{f_std:.3f}"
        print(f"{method_name:<12} {class_name:<12} "
              f"{p_text:>18} {r_text:>18} {f_text:>18}")

# Save complete 

In [ ]:
summary_rows = []
detail_rows = []
history_rows = []

for method_name, fold_results in all_results.items():
    summary_row = {'method': method_name, 'n_outer_runs': len(fold_results)}
    for key, _ in METRICS_TO_SHOW:
        mean, std = _mean_std(fold_results, key)
        summary_row[f'{key}_mean'] = mean
        summary_row[f'{key}_std'] = std
    summary_rows.append(summary_row)

    for result in fold_results:
        detail_rows.append({k: v for k, v in result.items() if k != 'history'})
        for h in result.get('history', []):
            history_rows.append({
                'method': method_name,
                'repeat': result['repeat'],
                'fold_in_repeat': result['fold_in_repeat'],
                'outer_run': result['outer_run'],
                **h,
            })

summary_df = pd.DataFrame(summary_rows)
diagnostics_df = pd.DataFrame(detail_rows).sort_values(
    ['method', 'repeat', 'fold_in_repeat']
).reset_index(drop=True)
history_df = pd.DataFrame(history_rows).sort_values(
    ['method', 'repeat', 'fold_in_repeat', 'epoch']
).reset_index(drop=True)

summary_path = Path(str(OUTPUT_PREFIX) + '_summary.csv')
diagnostics_path = Path(str(OUTPUT_PREFIX) + '_fold_diagnostics.csv')
history_path = Path(str(OUTPUT_PREFIX) + '_training_history.csv')

summary_df.to_csv(summary_path, index=False)
diagnostics_df.to_csv(diagnostics_path, index=False)
history_df.to_csv(history_path, index=False)

print(f"Saved summary:      {summary_path}")
print(f"Saved diagnostics:  {diagnostics_path}")
print(f"Saved history:      {history_path}")

AUDIT_COLUMNS = [
    'method', 'repeat', 'fold_in_repeat', 'outer_run', 'model_seed',
    'n_train_patients', 'n_val_patients', 'n_test_patients',
    'n_method_train_windows', 'n_method_val_windows', 'n_canonical_test_windows',
    'best_epoch', 'epochs_ran', 'threshold_used',
    'train_auroc', 'train_pr_auc', 'train_f1_hypo',
    'val_auroc', 'val_pr_auc', 'val_f1_hypo',
    'test_auroc', 'test_pr_auc', 'test_f1_hypo',
]
print("\nPER-FOLD AUDIT")
display(diagnostics_df[[c for c in AUDIT_COLUMNS if c in diagnostics_df.columns]])

# Explicitly aligned exploratory paired comparisons

In [ ]:
from scipy.stats import wilcoxon


def results_by_key(results):
    return {
        (int(r['repeat']), int(r['fold_in_repeat'])): r
        for r in results
    }


def aligned_arrays(a_results, b_results, metric):
    A = results_by_key(a_results)
    B = results_by_key(b_results)
    keys = sorted(set(A) & set(B))
    keys = [key for key in keys
            if np.isfinite(A[key].get(metric, np.nan))
            and np.isfinite(B[key].get(metric, np.nan))]
    a = np.asarray([A[key][metric] for key in keys], dtype=float)
    b = np.asarray([B[key][metric] for key in keys], dtype=float)
    return keys, a, b


def exploratory_signed_rank(a, b, alternative='greater'):
    if len(a) < 2 or len(a) != len(b):
        return np.nan, np.nan
    if np.allclose(a, b):
        return 0.0, 1.0
    try:
        statistic, p_value = wilcoxon(
            a, b, alternative=alternative, zero_method='wilcox'
        )
        return float(statistic), float(p_value)
    except ValueError:
        return np.nan, np.nan


pidck_results = all_results['pidck']
print("\n" + "=" * 105)
print("PIDCK VS OTHER METHODS — aligned by (repeat, fold)")
print("Fold-level Wilcoxon values are exploratory because repeated-CV folds overlap.")
print("Primary interpretation should emphasize paired effect sizes and distributions.")
print("=" * 105)
print(f"{'Comparator':<14} {'Pairs':>5} {'dF1':>10} {'pF1*':>10} "
      f"{'dPR-AUC':>10} {'pPR*':>10} {'dAUROC':>10} {'pAUC*':>10}")
print('-' * 105)

for method_name, comparator_results in all_results.items():
    if method_name == 'pidck':
        continue

    keys_f1, pidck_f1, comp_f1 = aligned_arrays(
        pidck_results, comparator_results, 'f1_hypo'
    )
    keys_pr, pidck_pr, comp_pr = aligned_arrays(
        pidck_results, comparator_results, 'pr_auc'
    )
    keys_auc, pidck_auc, comp_auc = aligned_arrays(
        pidck_results, comparator_results, 'auroc'
    )

    _, p_f1 = exploratory_signed_rank(pidck_f1, comp_f1)
    _, p_pr = exploratory_signed_rank(pidck_pr, comp_pr)
    _, p_auc = exploratory_signed_rank(pidck_auc, comp_auc)

    print(
        f"{method_name:<14} {len(keys_f1):>5} "
        f"{np.mean(pidck_f1-comp_f1):>+10.4f} {p_f1:>10.4f} "
        f"{np.mean(pidck_pr-comp_pr):>+10.4f} {p_pr:>10.4f} "
        f"{np.mean(pidck_auc-comp_auc):>+10.4f} {p_auc:>10.4f}"
    )

print("\n* Exploratory only; do not describe the 15 folds as independent subjects.")

# Final runtime integrity checks

In [ ]:
expected_keys = {
    (split['repeat'], split['fold_in_repeat']) for split in OUTER_SPLITS
}

for method_name, fold_results in all_results.items():
    actual_keys = {
        (r['repeat'], r['fold_in_repeat']) for r in fold_results
    }
    if actual_keys != expected_keys:
        raise RuntimeError(
            f"{method_name}: final fold-key mismatch. "
            f"Missing={sorted(expected_keys-actual_keys)}, "
            f"extra={sorted(actual_keys-expected_keys)}"
        )

# Canonical test counts must be identical across methods within every outer run.
counts = diagnostics_df.pivot(
    index='outer_run', columns='method', values='n_canonical_test_windows'
)
if not (counts.nunique(axis=1) == 1).all():
    raise RuntimeError("Canonical test-window counts differ across methods")

# Patient assignments must be identical across methods within each outer run.
for field in ('train_patient_ids', 'val_patient_ids', 'test_patient_ids'):
    pivot = diagnostics_df.pivot(index='outer_run', columns='method', values=field)
    if not (pivot.nunique(axis=1) == 1).all():
        raise RuntimeError(f"{field} differs across methods")

print("\nFINAL INTEGRITY CHECK PASSED")
print("  - Standard FCN used for every method")
print("  - 15/15 aligned outer runs completed for every method")
print("  - Same train/validation/test patient assignments across methods")
print("  - Method-specific eligible training/validation inputs")
print("  - All targets from originally observed future glucose")
print("  - Exact same canonical observed test arrays across methods")
print("  - No window-level fallback")
print("  - Effective validation-based early stopping and threshold selection")
print("  - Scope: offline downstream imputation benchmark, not causal real-time deployment")

# FINAL STATISTICS

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import t as student_t


# Align results by (repeat, fold)

def results_by_key(results):
    return {
        (int(r['repeat']), int(r['fold_in_repeat'])): r
        for r in results
    }


def aligned_metric_arrays(a_results, b_results, metric):
    A = results_by_key(a_results)
    B = results_by_key(b_results)

    keys = sorted(set(A) & set(B))

    valid_keys = []
    a_vals = []
    b_vals = []

    for key in keys:
        av = A[key].get(metric, np.nan)
        bv = B[key].get(metric, np.nan)

        if np.isfinite(av) and np.isfinite(bv):
            valid_keys.append(key)
            a_vals.append(float(av))
            b_vals.append(float(bv))

    return (
        valid_keys,
        np.asarray(a_vals, dtype=float),
        np.asarray(b_vals, dtype=float)
    )


# Nadeau-Bengio corrected repeated-CV paired t-test
# --------------------------------------------------------------------
def corrected_repeated_cv_test(
    method_a,
    method_b,
    n_splits=5,
    confidence=0.95
):
    """
    method_a and method_b:
        paired metric arrays from identical repeated-CV folds.

    Difference is defined as:
        method_a - method_b

    Correction:
        Var(mean difference) ≈
        (1/n + n_test/n_train) * sample_variance

    For k-fold CV:
        n_test/n_train ≈ 1 / (k - 1)

    With k=5:
        ratio = 1/4 = 0.25
    """

    a = np.asarray(method_a, dtype=float)
    b = np.asarray(method_b, dtype=float)

    if len(a) != len(b):
        raise ValueError("Arrays must be paired and equal length.")

    if len(a) < 2:
        return {
            'n': len(a),
            'mean_difference': np.nan,
            'corrected_se': np.nan,
            't_statistic': np.nan,
            'p_two_sided': np.nan,
            'ci_low': np.nan,
            'ci_high': np.nan,
        }

    differences = a - b

    n = len(differences)

    mean_diff = float(np.mean(differences))
    variance = float(np.var(differences, ddof=1))

    # Approximate test/train ratio for k-fold CV
    test_train_ratio = 1.0 / (n_splits - 1)

    correction = (1.0 / n) + test_train_ratio

    corrected_se = np.sqrt(correction * variance)

    if corrected_se == 0:
        t_stat = 0.0
        p_two = 1.0
        ci_low = mean_diff
        ci_high = mean_diff
    else:
        t_stat = mean_diff / corrected_se
        df = n - 1

        p_two = 2.0 * student_t.sf(abs(t_stat), df=df)

        alpha = 1.0 - confidence
        t_crit = student_t.ppf(1.0 - alpha / 2.0, df=df)

        ci_low = mean_diff - t_crit * corrected_se
        ci_high = mean_diff + t_crit * corrected_se

    return {
        'n': n,
        'mean_difference': mean_diff,
        'corrected_se': float(corrected_se),
        't_statistic': float(t_stat),
        'p_two_sided': float(p_two),
        'ci_low': float(ci_low),
        'ci_high': float(ci_high),
    }


# --------------------------------------------------------------------
# Holm correction

def holm_adjust(p_values):
    """
    Holm step-down family-wise error correction.
    Returns adjusted p-values in original order.
    """

    p_values = np.asarray(p_values, dtype=float)

    adjusted = np.full(len(p_values), np.nan)

    valid_idx = np.where(np.isfinite(p_values))[0]

    if len(valid_idx) == 0:
        return adjusted

    valid_p = p_values[valid_idx]

    order = np.argsort(valid_p)
    ordered_p = valid_p[order]

    m = len(ordered_p)

    ordered_adj = np.empty(m)

    running_max = 0.0

    for rank, p in enumerate(ordered_p):
        value = (m - rank) * p
        running_max = max(running_max, value)
        ordered_adj[rank] = min(running_max, 1.0)

    # restore original order among valid p-values
    inverse_order = np.empty(m, dtype=int)
    inverse_order[order] = np.arange(m)

    valid_adjusted = ordered_adj[inverse_order]

    adjusted[valid_idx] = valid_adjusted

    return adjusted


# --------------------------------------------------------------------
#  Win / tie / loss

def win_tie_loss(a, b, tolerance=1e-6):
    diff = np.asarray(a) - np.asarray(b)

    wins = int(np.sum(diff > tolerance))
    losses = int(np.sum(diff < -tolerance))
    ties = int(len(diff) - wins - losses)

    return wins, ties, losses


# --------------------------------------------------------------------
#  PIDCK vs all comparators

PRIMARY_METRICS = {
    'f1_hypo': 'F1 hypo',
    'pr_auc': 'PR-AUC',
    'auroc': 'AUROC',
}

comparators = ['original', 'brits', 'csdi']

rows = []

for comparator in comparators:

    for metric, metric_label in PRIMARY_METRICS.items():

        keys, pidck_values, comparator_values = aligned_metric_arrays(
            all_results['pidck'],
            all_results[comparator],
            metric
        )

        stats = corrected_repeated_cv_test(
            pidck_values,
            comparator_values,
            n_splits=N_SPLITS,
            confidence=0.95
        )

        wins, ties, losses = win_tie_loss(
            pidck_values,
            comparator_values
        )

        rows.append({
            'comparison': f'PIDCK vs {comparator.upper()}',
            'metric': metric_label,
            'pairs': len(keys),

            'PIDCK_mean': float(np.mean(pidck_values)),
            'Comparator_mean': float(np.mean(comparator_values)),

            'mean_difference': stats['mean_difference'],

            'CI95_low': stats['ci_low'],
            'CI95_high': stats['ci_high'],

            'corrected_t': stats['t_statistic'],
            'p_raw': stats['p_two_sided'],

            'wins': wins,
            'ties': ties,
            'losses': losses,
        })


stats_df = pd.DataFrame(rows)


# --------------------------------------------------------------------
#  Holm correction across the 9 planned tests

stats_df['p_holm'] = holm_adjust(
    stats_df['p_raw'].to_numpy()
)


# --------------------------------------------------------------------
# Significance indicator

stats_df['holm_significant_0.05'] = (
    stats_df['p_holm'] < 0.05
)


# --------------------------------------------------------------------
# print

display_cols = [
    'comparison',
    'metric',
    'pairs',
    'PIDCK_mean',
    'Comparator_mean',
    'mean_difference',
    'CI95_low',
    'CI95_high',
    'corrected_t',
    'p_raw',
    'p_holm',
    'wins',
    'ties',
    'losses',
    'holm_significant_0.05'
]

print("\n" + "=" * 130)
print("CORRECTED REPEATED-CV STATISTICAL ANALYSIS")
print("Nadeau-Bengio variance correction + Holm multiple-testing correction")
print("=" * 130)

display(
    stats_df[display_cols].round({
        'PIDCK_mean': 4,
        'Comparator_mean': 4,
        'mean_difference': 4,
        'CI95_low': 4,
        'CI95_high': 4,
        'corrected_t': 3,
        'p_raw': 4,
        'p_holm': 4,
    })
)


# --------------------------------------------------------------------
# Save results

corrected_stats_path = Path(
    str(OUTPUT_PREFIX) + '_corrected_repeated_cv_statistics.csv'
)

stats_df.to_csv(corrected_stats_path, index=False)

print(f"\nSaved corrected statistics to:\n{corrected_stats_path}")


# --------------------------------------------------------------------
# Interpretation helper

print("\nINTERPRETATION")
print("-" * 70)

for _, row in stats_df.iterrows():

    direction = (
        "higher"
        if row['mean_difference'] > 0
        else "lower"
        if row['mean_difference'] < 0
        else "equal"
    )

    print(
        f"{row['comparison']} | {row['metric']}: "
        f"PIDCK {direction} by {row['mean_difference']:+.4f}; "
        f"95% corrected CI "
        f"[{row['CI95_low']:+.4f}, {row['CI95_high']:+.4f}]; "
        f"Holm p={row['p_holm']:.4f}; "
        f"W/T/L={row['wins']}/{row['ties']}/{row['losses']}"
    )